<a href="https://colab.research.google.com/github/annabzsantos/recsys-livros-filmes/blob/main/trabalhofinal_deep.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
!pip install -q -U transformers accelerate bitsandbytes

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.7/10.7 MB 73.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 383.7/383.7 kB 22.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 12.3 MB/s eta 0:00:00


In [1]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, pipeline

# Trocamos para um modelo de 3 Bilhões de parâmetros (muito mais leve e estável)
modelo_id = "Qwen/Qwen2.5-3B-Instruct"

print("Baixando e carregando o modelo (isso vai caber com folga na RAM e VRAM!)...")
tokenizer = AutoTokenizer.from_pretrained(modelo_id)

# Carregando direto na GPU em float16, sem precisar de quantização pesada
modelo = AutoModelForCausalLM.from_pretrained(
    modelo_id,
    device_map="auto",
    torch_dtype=torch.float16,
    low_cpu_mem_usage=True # Otimiza a passagem da RAM para a VRAM
)

# Criando o pipeline de geração
gerador = pipeline("text-generation", model=modelo, tokenizer=tokenizer)

def recomendar_midia_alinhada(pedido_usuario):
    # Estruturando as mensagens no formato de chat (lista de dicionários)
    mensagens = [
        {"role": "system", "content": """Você é um curador cultural empático e objetivo. Siga ESTAS REGRAS rigorosamente:
1. Analise o pedido para identificar se o usuário quer LIVROS ou FILMES. Responda apenas com a mídia solicitada.
2. Identifique o gênero, o estado emocional ou o objetivo de aprendizado.
3. Forneça EXATAMENTE entre 3 e 5 sugestões. Nem menos, nem mais.
4. Formate cada sugestão usando a seguinte estrutura:
   - **Título** (Ano)
   - **Por que esta é uma boa escolha:** (Uma breve explicação conectando a obra diretamente ao pedido ou sentimento do usuário).
5. Mantenha um tom amigável, mas vá direto ao ponto nas recomendações."""},
        {"role": "user", "content": pedido_usuario}
    ]

    # O tokenizer aplica a formatação correta (ChatML) que o Qwen exige automaticamente
    prompt = tokenizer.apply_chat_template(mensagens, tokenize=False, add_generation_prompt=True)

    print(f"\n[Gerando recomendações...]")
    resposta = gerador(
        prompt,
        max_new_tokens=600,
        temperature=0.4,
        do_sample=True,
        return_full_text=False # Garante que a saída contenha apenas a resposta do bot, sem repetir o seu prompt!
    )

    return resposta[0]['generated_text'].strip()

# Simulando as interações com os seus exemplos
exemplo_1 = "Hoje estou triste, quero indicação para um filme de comédia romântica"
exemplo_2 = "Quero começar na área de analise de dados, me indique livros para saber por onde começar"

print("\n--- Teste 1: O dia triste ---")
print(recomendar_midia_alinhada(exemplo_1))

print("\n--- Teste 2: Foco nos estudos ---")
print(recomendar_midia_alinhada(exemplo_2))

Baixando e carregando o modelo (isso vai caber com folga na RAM e VRAM!)...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/661 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

...

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

Passing `generation_config` together with generation-related arguments=({'temperature', 'do_sample', 'max_new_tokens'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
Both `max_new_tokens` (=600) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



--- Teste 1: O dia triste ---

[Gerando recomendações...]


Both `max_new_tokens` (=600) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


- **A Vida Invisível (2019)**
  - **Por que esta é uma boa escolha:** Este filme retrata uma história romântica leve e hilária, combinando elementos de comédia romântica com uma pitada de drama, o que pode ajudar a aliviar seu humor e proporcionar um momento de descontração.

- **Meu Malvado Favorito (2001)**
  - **Por que esta é uma boa escolha:** Comédias românticas divertidas e emocionantes, este filme oferece momentos românticos envolventes e cenas engraçadas que podem ajudar a melhorar seu humor.

- **Um Sonho em Nova York (2004)**
  - **Por que esta é uma boa escolha:** Uma mistura perfeita de comédia romântica e drama, este filme conta uma história de amor apaixonante que pode ser cativante e refrescante para quem está precisando de um pouco de diversão.

- **O Diário de Bridget Jones (2001)**
  - **Por que esta é uma boa escolha:** Esta série de filmes retrata uma história de amor bem-humorada e autodeprecada, que pode ajudar a aliviar sua tristeza através da risada e da reflex